# Наблюдение за CTF-агентом
Запустите runner: `python -m pipeline --config config/pipeline.json`. Закрытие вкладки не останавливает прогон. Чтобы вернуться, укажите сохранённый `RUN_ID` в ячейке наблюдения, **не запускайте повторно** ячейку создания прогона. Вывод агента может содержать флаг — не публикуйте ноутбук без проверки.

In [ ]:
import importlib, json, os, sys, urllib.request
from pathlib import Path
from IPython.display import display, Markdown, HTML

ROOT = Path.cwd() if (Path.cwd() / 'pipeline').is_dir() else Path.cwd().parent
if not (ROOT / 'pipeline').is_dir():
    raise RuntimeError('Откройте ноутбук из корня проекта или каталога notebooks')
sys.path.insert(0, str(ROOT))
import pipeline.notebook
Observer = importlib.reload(pipeline.notebook).Observer

BASE = os.environ.get('PIPELINE_URL', 'http://127.0.0.1:8765')
def api(path, method='GET', body=None):
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(BASE + path, data=data, method=method, headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=300 if path.endswith('/reevaluate') else 30) as response:
        return json.load(response)

api('/health')

## Новый прогон
Выполняйте только для новой попытки: эта ячейка запускает модель.

In [ ]:
overrides = {}  # Например: {'model': 'gpt-5.6-sol', 'thinking': 'low'}
started = api('/runs', 'POST', overrides)
RUN_ID = started['run_id']
print('Сохраните RUN_ID:', RUN_ID)

## Наблюдение
Каждый вызов показывает текущий шаг, хронологию, проверки и бюджеты в течение 20 секунд, затем возвращает управление ядру. Повторите для обновления. `RUN_DIR` — путь к локальному пакету: он позволяет читать сохранённые события и неполные свидетельства без сервера и без итогового отчёта. Сохранённый running без живого владельца не подтверждает, что прогон продолжается.

In [ ]:
# RUN_ID = 'идентификатор из истории'
RUN_DIR = ROOT / 'runs' / RUN_ID  # При другом runs_dir укажите фактический путь
if ('observer' not in globals() or not isinstance(observer, Observer)
        or observer.run_id != RUN_ID or observer.run_dir != RUN_DIR):
    observer = Observer(api, RUN_ID, run_dir=RUN_DIR)
observer.watch(seconds=20)
display(HTML(observer.details_html()))  # Полные сообщения, команды, ответы и ссылки раскрываются по запросу

## Остановка
Запрашивает штатное завершение прогона. Ядро доступно после окончания 20-секундного наблюдения.

In [ ]:
api(f'/runs/{RUN_ID}/stop', 'POST', {})

## Полный отчёт
После завершения — независимые результаты проверок, попытки replay и основания вердикта. Отсутствие отчёта само по себе не означает инфраструктурный сбой. Для чтения только сохранённых данных используйте `Observer(None, RUN_ID, run_dir=RUN_DIR)` в ячейке наблюдения. Журнал observations.md — отдельные наблюдения агента, не доказательства.

In [ ]:
report = api(f"/runs/{RUN_ID}/report")
display(Markdown(api(f"/runs/{RUN_ID}/report-markdown")["markdown"]))
report["solution"]

In [ ]:
report = api(f"/runs/{RUN_ID}/reevaluate", "POST", {})
display(Markdown(api(f"/runs/{RUN_ID}/report-markdown")["markdown"]))

## История
Скопируйте `run_id` в ячейку наблюдения для открытия сохранённой хронологии.

In [ ]:
from html import escape
from IPython.display import HTML
rows = ['<table><tr><th>Прогон</th><th>Статус</th><th>Стадия</th><th>Причина завершения</th></tr>']
for run in api('/runs'):
    values = [run['run_id'], run['status'], run.get('stage'), (run.get('stop_reason') or {}).get('kind', '')]
    rows.append('<tr>' + ''.join('<td>' + escape(str(v)) + '</td>' for v in values) + '</tr>')
display(HTML(''.join(rows) + '</table>'))